# Zero-Shot CLIP Classification for Vector Features

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/clip_classify_vector.ipynb)

This notebook demonstrates how to classify vector polygon features using zero-shot CLIP inference with the `geoai.clip_classify` module. Given a set of polygons and a raster image, CLIP extracts an image chip for each polygon and matches it against user-provided category labels — no training data required.

## Key Features

- **Zero-shot classification**: Just provide candidate labels, no training needed
- **Automatic CRS alignment**: Vector and raster CRS are aligned automatically
- **Batch GPU inference**: Efficient batch processing with GPU acceleration
- **Top-k predictions**: Get multiple ranked predictions per polygon
- **Multiple export formats**: Save results as GeoJSON, GeoParquet, or GeoPackage

## Install packages

In [ ]:
# %pip install geoai-py

## Import libraries

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

from geoai import clip_classify_vector, CLIPVectorClassifier, download_file

## Download sample data

We use a NAIP aerial image and building footprint polygons hosted on Hugging Face. The raster is a high-resolution RGB image and the vector file contains building outlines extracted from the same area.

In [ ]:
raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train.tif"
)
vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"

raster_path = download_file(raster_url)
vector_path = download_file(vector_url)

## Preview the data

Load the vector polygons and display the raster with overlaid building footprints.

In [ ]:
gdf = gpd.read_file(vector_path)
print(f"Number of polygons: {len(gdf)}")
print(f"CRS: {gdf.crs}")
gdf.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)
    gdf_proj = gdf.to_crs(src.crs)
gdf_proj.boundary.plot(ax=ax, color="red", linewidth=0.5)
ax.set_title("NAIP Image with Building Footprints")
plt.tight_layout()
plt.show()

## Classify with the convenience function

The `clip_classify_vector` function is the simplest way to run zero-shot classification. Just provide:

1. **Vector data** — a file path or GeoDataFrame of polygons
2. **Raster path** — the image to extract chips from
3. **Labels** — candidate category names (no training required!)

CLIP encodes each image chip and compares it against the text embeddings of the labels to find the best match.

In [ ]:
labels = [
    "residential building",
    "commercial building",
    "industrial building",
    "vegetation",
    "parking lot",
]

result = clip_classify_vector(
    vector_data=vector_path,
    raster_path=raster_path,
    labels=labels,
)

## Explore results

The returned GeoDataFrame has two new columns:
- `clip_label` — the top-1 predicted category
- `clip_confidence` — the softmax confidence score (0 to 1)

In [ ]:
result[["geometry", "clip_label", "clip_confidence"]].head(10)

In [ ]:
print("Label distribution:")
print(result["clip_label"].value_counts())
print(f"\nMean confidence: {result['clip_confidence'].mean():.3f}")

## Visualize classified polygons

Color-code each polygon by its predicted label and overlay on the raster.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)

result.plot(
    column="clip_label",
    ax=ax,
    legend=True,
    alpha=0.5,
    edgecolor="black",
    linewidth=0.3,
    legend_kwds={"loc": "upper left", "fontsize": 8},
)
ax.set_title("Zero-Shot CLIP Classification Results")
plt.tight_layout()
plt.show()

## Visualize confidence scores

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
with rasterio.open(raster_path) as src:
    show(src, ax=ax)

result.plot(
    column="clip_confidence",
    ax=ax,
    legend=True,
    alpha=0.6,
    edgecolor="black",
    linewidth=0.3,
    cmap="RdYlGn",
    legend_kwds={"shrink": 0.5, "label": "Confidence"},
)
ax.set_title("CLIP Confidence Scores")
plt.tight_layout()
plt.show()

## Using the class API with top-k predictions

For more control, use `CLIPVectorClassifier` directly. Setting `top_k=3` returns the top 3 predictions per polygon, useful for understanding label ambiguity.

In [ ]:
classifier = CLIPVectorClassifier()

result_topk = classifier.classify(
    vector_data=vector_path,
    raster_path=raster_path,
    labels=labels,
    top_k=3,
    batch_size=32,
)

In [ ]:
# Show top-3 predictions for the first 5 classified polygons
classified = result_topk.dropna(subset=["clip_label"])
for i in range(min(5, len(classified))):
    row = classified.iloc[i]
    print(f"Polygon {classified.index[i]}:")
    for label, score in zip(row["clip_top_k_labels"], row["clip_top_k_scores"]):
        print(f"  {label:30s} {score:.3f}")
    print()

## Export results

Save the classified GeoDataFrame to GeoJSON, GeoParquet, or GeoPackage.

In [ ]:
result.to_file("classified_buildings.geojson", driver="GeoJSON")
print("Results saved to classified_buildings.geojson")

## Summary

This notebook demonstrated:

1. **Zero-shot classification**: Classifying building polygons with no training data — just candidate labels
2. **Convenience function**: Using `clip_classify_vector()` for single-call classification
3. **Class API**: Using `CLIPVectorClassifier` for reusable model instances and top-k predictions
4. **Visualization**: Color-coded polygon maps and confidence score overlays
5. **Export**: Saving annotated results to standard geospatial formats

### Key Parameters

| Parameter | Description | Default |
|-----------|-------------|----------|
| `labels` | Candidate category names | (required) |
| `label_prefix` | Text prefix for CLIP encoding | `"a satellite image of "` |
| `model_name` | Hugging Face CLIP model ID | `"openai/clip-vit-base-patch32"` |
| `top_k` | Number of top predictions per polygon | `1` |
| `batch_size` | Images per inference batch | `16` |
| `min_chip_size` | Minimum chip dimension in pixels | `10` |
| `output_path` | Path to save results (`.geojson`, `.parquet`, `.gpkg`) | `None` |

### Next Steps

- Try different CLIP models (e.g., `openai/clip-vit-large-patch14`) for higher accuracy
- Experiment with custom `label_prefix` values for domain-specific prompts
- Adjust `min_chip_size` based on polygon sizes in your dataset
- Use top-k predictions to identify ambiguous or misclassified features